# Algoritmo Genético para el Manufacturing Cell Design Problem (MCDP)

**Objetivo:** Minimizar el _Costo de Transporte Intercelular_ (Adaptación) agrupando máquinas en celdas manufactureras.

$$GE = \frac{e_{in}}{e_{in} + e_{out} + e_{voids}}$$

- $e_{in}$: número de 1s dentro de bloques diagonales (intra-celda)
- $e_{out}$: número de 1s fuera de los bloques (elementos excepcionales)
- $e_{voids}$: número de 0s dentro de los bloques diagonales (vacíos)

**Convención de la matriz:** filas = máquinas, columnas = partes.

## 1. Setup e imports

In [ ]:
import sys
import os

# Clonar repositorio si estamos en Google Colab
if 'google.colab' in sys.modules:
    if not os.path.exists("Tarea-2-Optimizacion"):
        !git clone https://github.com/felipe-astudillo-s/Tarea-2-Optimizacion.git
    %cd Tarea-2-Optimizacion


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mcdp_ga import (
    MCDPInstance,
    GeneticAlgorithm,
    ExperimentRunner,
    parse_instance_file,
)

print("Imports OK")

## 2. Cargar instancias

In [ ]:
INSTANCE_FILE_1 = os.path.join("Instancias", "instanciasMCDP.txt")
INSTANCE_FILE_2 = os.path.join("Instancias", "instancia_test_grande.txt")

instances_list = parse_instance_file(INSTANCE_FILE_1) + parse_instance_file(INSTANCE_FILE_2)
instance_names = ["Instancia 1 (pequeña)", "Boctor 1", "Boctor 10", "Instancia Grande"]

instances = dict(zip(instance_names, instances_list))

for name, inst in instances.items():
    print(f"{name}: {inst.n_machines} máquinas × {inst.n_parts} partes "
          f"| C={inst.C}, MaxM={inst.max_m}, total_ones={inst.total_ones}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, inst) in zip(axes, instances.items()):
    ax.imshow(inst.matrix, cmap="Blues", aspect="auto")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Partes")     # columnas
    ax.set_ylabel("Máquinas")   # filas

plt.suptitle("Matrices de incidencia (azul = 1)", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Configuración de parámetros

Modifica aquí los parámetros del AG y el número de ejecuciones.

In [ ]:
# --- Parámetros del Algoritmo Genético ---
GA_CONFIG = {
    "objective"         : "cost",    # optimizar costo de transporte intercelular
    "pop_size"          : 50,      # tamaño de la población
    "max_generations"   : 93,     # máximo de generaciones por corrida (~4500 evaluaciones)
    "max_time_seconds"  : 9999.0,   # tiempo máximo por corrida (segundos)
    "tournament_size"          : 2,       # participantes por torneo
    "crossover_prob"          : 0.9,     # probabilidad de cruce
    "mutation_prob"          : 0.2,    # probabilidad de mutación
    "elite_count"          : 2,       # individuos de élite por generación
    "crossover_type"    : "two_point",  # "two_point" o "one_point"
    "random_seed"       : 0,       # semilla base (None = no determinístico)
}

N_RUNS = 10  # número de ejecuciones independientes por instancia

print("Configuración:")
for k, v in GA_CONFIG.items():
    print(f"  {k:20s} = {v}")
print(f"  {'N_RUNS':20s} = {N_RUNS}")

## 4. Ejecución de experimentos

> **Nota:** Con `max_time_seconds=120` y `N_RUNS=10`, el tiempo total estimado es ~40 min para las instancias grandes. Reducir `max_time_seconds` o `max_generations` para pruebas rápidas.

In [ ]:
runner = ExperimentRunner(ga_config=GA_CONFIG, n_runs=N_RUNS)
results = runner.run_all(instances)

print("\nEjecución finalizada.")

## 5. Tabla de resultados

In [ ]:
rows = []
for name, res in results.items():
    row = {"Instancia": name}
    for i, v in enumerate(res["fitness_values"], 1):
        row[f"Run {i}"] = round(v, 4)
    row["Media"] = round(res["mean"], 4)
    row["Desv. Std."] = round(res["std"], 4)
    row["Mejor"] = round(res["best_fitness"], 4)
    rows.append(row)

df = pd.DataFrame(rows).set_index("Instancia")
df.style.highlight_max(subset=["Mejor"], color="lightgreen").set_caption(
    "Grouping Efficacy (GE) – 10 ejecuciones independientes"
)

In [ ]:
print(df.to_string())

## 6. Box plots

In [ ]:
data   = [results[n]["fitness_values"] for n in instance_names]
labels = [n.replace(" ", "\n") for n in instance_names]

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(data, patch_artist=True, notch=False,
                medianprops=dict(color="black", linewidth=2))

colors = ["#4C72B0", "#DD8452", "#55A868"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(range(1, len(instance_names) + 1))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Grouping Efficacy (GE)", fontsize=11)
ax.set_title("Distribución de GE – 10 ejecuciones por instancia", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("boxplot_resultados.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: boxplot_resultados.png")

## 7. Curvas de convergencia (mejor corrida por instancia)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, name in zip(axes, instance_names):
    res = results[name]
    best_run_idx = int(np.argmax(res["fitness_values"]))
    curve = res["convergence_curves"][best_run_idx]

    ax.plot(curve, linewidth=1.5, color="steelblue")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Generación", fontsize=9)
    ax.set_ylabel("Mejor GE", fontsize=9)
    ax.grid(linestyle="--", alpha=0.4)
    ax.annotate(
        f"GE final: {curve[-1]:.4f}",
        xy=(len(curve) - 1, curve[-1]),
        xytext=(-60, -20),
        textcoords="offset points",
        fontsize=8,
        arrowprops=dict(arrowstyle="->", color="gray"),
    )

plt.suptitle("Convergencia del AG – mejor corrida por instancia", fontsize=12)
plt.tight_layout()
plt.savefig("convergencia.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: convergencia.png")

## 8. Mejor solución encontrada por instancia

Muestra la asignación de máquinas y partes a celdas, y visualiza la
matriz reorganizada en bloques diagonales.

**Convención:** filas = máquinas, columnas = partes.

In [ ]:
def print_solution(name: str, inst: MCDPInstance, sol: dict):
    ma = sol["best_machine_assignment"]
    pa = sol["best_part_assignment"]
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"  GE = {sol['best_fitness']:.4f}")
    print(f"{'='*55}")
    for c in range(inst.C):
        maq = sorted(int(x) + 1 for x in np.where(ma == c)[0])  # 1-indexed
        par = sorted(int(x) + 1 for x in np.where(pa == c)[0])
        print(f"  Celda {c+1}: máquinas {maq}")
        print(f"           partes   {par}")


def plot_reorganized(name: str, inst: MCDPInstance, sol: dict, ax):
    """Reordena filas (máquinas) y columnas (partes) según la asignación a celdas."""
    ma = sol["best_machine_assignment"]  # longitud n_machines
    pa = sol["best_part_assignment"]     # longitud n_parts

    # Ordenar máquinas (filas) por celda, luego partes (columnas) por celda
    row_order = np.argsort(ma, kind="stable")  # índices de máquinas agrupadas por celda
    col_order = np.argsort(pa, kind="stable")  # índices de partes agrupadas por celda

    reorg = inst.matrix[np.ix_(row_order, col_order)]
    ax.imshow(reorg, cmap="Blues", aspect="auto")
    ax.set_title(f"{name}\nGE={sol['best_fitness']:.4f}", fontsize=9)
    ax.set_xlabel("Partes (reordenadas)", fontsize=8)
    ax.set_ylabel("Máquinas (reordenadas)", fontsize=8)

    # Líneas de frontera entre celdas
    for c in range(inst.C - 1):
        row_split = int((ma == c).sum()) - 0.5  # frontera horizontal (entre máquinas)
        col_split = int((pa == c).sum()) - 0.5  # frontera vertical (entre partes)
        ax.axhline(row_split, color="red", linewidth=1.5)
        ax.axvline(col_split, color="red", linewidth=1.5)


fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, name in zip(axes, instance_names):
    sol = results[name]["best_solution"]
    inst = instances[name]
    print_solution(name, inst, sol)
    plot_reorganized(name, inst, sol, ax)

plt.suptitle(
    "Matrices reorganizadas por la mejor solución (línea roja = frontera de celda)",
    fontsize=11,
)
plt.tight_layout()
plt.savefig("matrices_reorganizadas.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: matrices_reorganizadas.png")

## Resumen final

In [ ]:
print("\nRESUMEN DE RESULTADOS")
print(f"{'Instancia':<25} {'Media GE':>10} {'Desv. Std.':>12} {'Mejor GE':>10}")
print("-" * 60)
for name, res in results.items():
    print(f"{name:<25} {res['mean']:>10.4f} {res['std']:>12.4f} {res['best_fitness']:>10.4f}")
print("\nParámetros utilizados:")
for k, v in GA_CONFIG.items():
    print(f"  {k} = {v}")
print(f"  N_RUNS = {N_RUNS}")